# 03 · Feature engineering (CP2)

Builds the model table for the retrospective experiment:

> does weather, hotspot detections and upwind exposure help predict the **CAMS model-field
> PM2.5 at t+6 h** at one grid cell?

What this notebook guarantees, and what it does not:

- It reads the CP1 output **from disk** (`data/interim/hourly_aligned.parquet` plus
  `audit_report.json`) and re-verifies both SHA256 checksums. It never uses variables left over
  from a notebook 02 kernel.
- The target is `target_cams_pm25_ug_m3_t+6h`, the **CAMS model field**, and `target_valid_at`
  records the exact hour it belongs to. CAMS is model output, not a station observation, and the
  snapshot has no `issued_at`, so nothing here shows operational forecast skill.
- Every feature at hour *t* only reads inputs dated at or before *t*. The target is the only
  column that looks forward, and no feature allowlist contains it.
- Missing hours are restored as `NaN` before any shift or rolling window, so a gap is never read
  as consecutive hours. CAMS gaps are never interpolated or filled with zero.

Ineligible rows stay in the table with `eligible = false` so the sample loss stays auditable:
the 48 h warm-up, the 6 h tail, and any row whose trailing 48 h window misses a `pm25_model`
feature (including CAMS at *t* itself).

In [ ]:
import pandas as pd

from pavana_ml.features.model_table import (
    build_model_table,
    load_feature_inputs,
    save_model_table,
)
from pavana_ml.paths import DATA_DIR

hourly, hotspots, inputs = load_feature_inputs(DATA_DIR)

print(f"interim table    : {inputs['hourly_aligned']['path']}")
print(f"  rows           : {inputs['hourly_aligned']['rows']}")
print(f"  sha256         : {inputs['hourly_aligned']['sha256']}")
print(f"hotspot cache    : {inputs['hotspots']['path']}")
print(f"  rows           : {inputs['hotspots']['rows']}")
print(f"  sha256         : {inputs['hotspots']['sha256']}")
print(f"audited at (UTC) : {inputs['hourly_aligned']['audited_at_utc']}")
snapshot = inputs["snapshot"]
print(f"snapshot         : {snapshot['revision']}")
print(f"  revision from  : {snapshot['revision_source']}")
print()
print("Both checksums match the audit report, so the features describe the audited data.")

## Build the table

Feature groups (all past-only at the origin hour *t*):

| Group | Content |
| --- | --- |
| `pm25_model` | CAMS PM2.5 at `t`; lags 1/3/6/12/24/48 h; means 6/24 h ending at `t`; 3 h change |
| `weather` | temperature, humidity, rain, wind speed, gust, pressure, mixing height, radiation |
| `wind` | flow-to east/north components, converted from meteorological FROM |
| `time` | UTC month, hour, and sin/cos of hour and day-of-year |
| `detection` | detection record counts and FRP sums within 50/100/300 km over 6/24/48 h, window `[t-W, t]` |
| `upwind` | exposure score, λ = 50 km, τ = 24 h, max radius 300 km, 48 h window |

The detection columns come from a sorted timestamp array with `searchsorted` plus prefix sums, so
the 1.16-million-row cache is not rescanned for each of the 24 thousand hours.

In [ ]:
table, metadata = build_model_table(hourly, hotspots, inputs=inputs)
eligibility = metadata["eligibility"]

print(f"rows             : {eligibility['rows']}")
print(f"columns          : {len(table.columns)}")
print(f"target           : {metadata['target']}  (kind: {metadata['target_kind']})")
print(f"eligible rows    : {eligibility['eligible']}")
expected = len(table) - 48 - metadata["horizon_hours"]
print(f"expected by plan : {expected}  (rows - 48 h warm-up - 6 h tail)")
print(
    f"excluded         : {eligibility['excluded_missing_target']} missing target at t+6, "
    f"{eligibility['excluded_incomplete_pm25_history']} incomplete PM2.5 history, "
    f"{eligibility['excluded_both']} both"
)
origins = metadata["eligible_origin_span_utc"]
targets = metadata["target_valid_span_utc"]
print(f"eligible origins : {origins['first']} .. {origins['last']}")
print(f"target valid span: {targets['first']} .. {targets['last']}")

## Checks worth doing by hand

The target column must be the CAMS value whose timestamp is exactly `target_valid_at`, and the
allowlists handed to CP3 must not contain the target or anything derived from the future.

In [ ]:
origin = pd.Timestamp("2026-06-01 00:00", tz="UTC")
row = table.loc[table["valid_at"] == origin].iloc[0]
cams_at_target = table.loc[table["valid_at"] == row["target_valid_at"], "cams_pm25_ug_m3"].iloc[0]

print(f"origin t        : {row['valid_at']}")
print(f"target_valid_at : {row['target_valid_at']}  (+{metadata['horizon_hours']} h)")
print(f"target value    : {row[metadata['target']]}")
print(f"CAMS at t+6     : {cams_at_target}")

assert row["target_valid_at"] - row["valid_at"] == pd.Timedelta(hours=metadata["horizon_hours"])
assert row[metadata["target"]] == cams_at_target
assert (table["target_valid_at"] - table["valid_at"] == pd.Timedelta(hours=6)).all()

horizon = metadata["horizon_hours"]
shifted = table[metadata["target"]].to_numpy()[:-horizon]
source = table["cams_pm25_ug_m3"].to_numpy()[horizon:]
print(f"whole column equals CAMS shifted by {horizon} h: {bool((shifted == source).all())}")
print(f"tail rows without a target: {int(table[metadata['target']].isna().sum())}")

In [ ]:
for name, columns in metadata["feature_groups"].items():
    print(f"{name:>10} ({len(columns):>2}): {', '.join(columns)}")
print()
for name, columns in metadata["feature_allowlists"].items():
    print(f"allowlist {name}: {len(columns)} columns")

leaked = sorted(
    column
    for columns in metadata["feature_allowlists"].values()
    for column in columns
    if column in (metadata["target"], metadata["target_valid_column"], "eligible")
)
assert not leaked, leaked
print()
print("No allowlist contains the target, target_valid_at or eligible.")

In [ ]:
missing = {c: n for c, n in metadata["feature_nan_counts_in_eligible"].items() if n}
print("Features that are NaN in some eligible rows (kept as NaN, never filled):")
for column, count in missing.items():
    print(f"  {column:>26}: {count}")
if not missing:
    print("  none")

window = "hotspot_records_n_300km_48h"
eligible = table["eligible"]
zero_share = float((table.loc[eligible, window] == 0).mean())
upwind = table.loc[eligible, "upwind_score"]
print()
print(f"eligible hours with zero detections in {window}: {zero_share:.1%}")
print("Zero means no detection record in this cached window: not proof of satellite coverage and")
print("not proof that no fire burned.")
print()
print(
    f"upwind_score on eligible rows: min {upwind.min():.3f}, median {upwind.median():.3f}, "
    f"max {upwind.max():.3f}, share at zero {float((upwind == 0).mean()):.1%}"
)
print("The score is an indicator, not a plume model and not a probability that smoke arrives.")

In [ ]:
saved = save_model_table(table, metadata, DATA_DIR)

print(f"table    : {saved['table_path']}  ({saved['table_rows']} rows)")
print(f"sha256   : {saved['table_sha256']}")
print(f"metadata : {saved['metadata_path']}")
print(
    f"status   : research_only={saved['research_only']}, "
    f"operational_ready={saved['operational_ready']}, "
    f"observation_validated={saved['observation_validated']}"
)
print()
print("CP3 (not built yet) must reject any table or metadata whose checksum does not match.")

## Guardrails for whoever reads this table next

- The target is the **CAMS model field**, not observed PM2.5. Any score computed on it is a
  retrospective comparison against a model field, not station validation.
- `upwind_score` combines FRP, distance decay, wind alignment and recency. It is not a plume
  model, not a probability, and a zero does not mean clean air.
- Detection counts are records in this cache. They are not confirmed fires and not a count of
  fire events; the sensor mix changes on 2026-08-01.
- `boundary_layer_height_m` keeps its missing hours as `NaN` and is never interpolated.
- Rows with `eligible = false` stay in the file on purpose: the 48 h warm-up, the 6 h tail and
  any row whose trailing 48 h window is incomplete cannot be scored, and hiding them would hide
  the reason.
- This table says nothing about operational skill until CP3 trains and evaluates on a held-out
  period.